# بناء chroma_v3 — Capital Legal Base  (resumable)

Embeds corpus **v3** (47,383 chunks) into a new vector store `chroma_v3`.

## This notebook survives a disconnect
The store is written **straight to your Google Drive**, and every chunk has a fixed ID.
If Colab drops you, just **re-run cell 5** — it detects what is already stored and carries
on from there. Nothing is embedded twice.

**Nothing is overwritten.** `chroma_v2` on your laptop is untouched. If v3 turns out worse,
you keep using v2.

### Before running
1. Upload `document_splits_v3.json.gz` to the **root of your Google Drive** (MyDrive).
2. **Runtime → Change runtime type → T4 GPU.**

### v2 vs v3
| | v2 | v3 |
|---|---|---|
| Laws | 588 | **3,075** |
| Cassation cases | 9,001 | 9,009 |
| Total chunks | 25,738 | **47,383** |

Adds the **Civil Code** (1,057 articles) and **Evidence Law** (158) — both previously
missing entirely — plus 709 amendments. Every legislation chunk is now prefixed with its
law name and chapter heading.

## 1. Check the GPU
Must list a GPU. If it errors: Runtime → Change runtime type → T4 GPU.

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q langchain-chroma langchain-huggingface sentence-transformers

## 3. Mount Drive and load the corpus
Expect **47383**. The assert stops the notebook if the file is wrong.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import gzip, json, collections
from langchain_core.documents import Document

SRC = '/content/drive/MyDrive/document_splits_v3.json.gz'
with gzip.open(SRC, 'rt', encoding='utf-8') as f:
    recs = json.load(f)

splits = [Document(page_content=r['page_content'], metadata=r['metadata']) for r in recs]
# Fixed IDs make the run idempotent: re-adding the same chunk cannot duplicate it.
ids = [f'c{i:06d}' for i in range(len(splits))]

print('chunks loaded:', len(splits))
print('by source    :', dict(collections.Counter(d.metadata['source'] for d in splits)))
assert len(splits) == 47383, f'expected 47383, got {len(splits)}'
print('OK')

## 4. Load the embedding model
Same model as v2 (`BAAI/bge-m3`) — required, or the new vectors would not be comparable to the old ones.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"batch_size": 64},
)
print('embedding model ready')

## 5. Embed → Drive  ← **re-run this cell if you get disconnected**

Writes to `MyDrive/chroma_v3`, so progress survives a dropped session.

On each run it asks the store how many chunks it already holds and starts from there.
Re-running after a disconnect resumes; re-running after completion does nothing.

Roughly **45–90 min** on a T4 (slower than local disk because Drive is network-mounted —
that is the price of being resumable).

Ignore red `Token indices sequence length is longer than...` warnings; that is the
tokenizer noting long chunks and is expected.

In [ ]:
import time
from langchain_chroma import Chroma

PERSIST = '/content/drive/MyDrive/chroma_v3'   # on Drive -> survives disconnects
BATCH   = 2000                                 # checkpoint every 2,000 chunks

vectordb = Chroma(persist_directory=PERSIST, embedding_function=embedding)
already = vectordb._collection.count()
print(f'already stored: {already:,} / {len(splits):,}')

if already >= len(splits):
    print('nothing to do - already complete')
else:
    t0 = time.time()
    for i in range(already, len(splits), BATCH):
        b_docs = splits[i:i + BATCH]
        b_ids  = ids[i:i + BATCH]
        vectordb.add_documents(documents=b_docs, ids=b_ids)
        done = i + len(b_docs)
        el   = time.time() - t0
        rate = (done - already) / el if el else 0
        eta  = (len(splits) - done) / rate / 60 if rate else 0
        print(f'{done:>6,}/{len(splits):,}   {el/60:5.1f} min elapsed   ~{eta:4.1f} min left',
              flush=True)
    print('EMBEDDING DONE in', round((time.time()-t0)/60, 1), 'min')

## 6. Verify — **the check that matters**

Must print **47383**. If it is lower, re-run cell 5 (it will resume). If it stays lower
after a full pass, tell Claude the number.

In [ ]:
count = vectordb._collection.count()
print('FINAL COUNT:', count, 'of', len(splits))
assert count == len(splits), f'MISMATCH: stored {count}, expected {len(splits)} - re-run cell 5'
print('OK - all chunks stored')

## 7. Smoke test — did this actually fix anything?

These are the questions the client's app failed on. We want to see **`lloc`** rows,
especially **`L1901`** (Civil Code) for عيوب الرضاء — previously that query returned
*zero* legislation, only court cases.

In [ ]:
tests = [
    'ما هي عيوب الرضاء في القانون المدني؟',
    'ما هو السبب في العقد؟',
    'ما هي عقوبة الرشوة؟',
    'ما هي قواعد الاثبات في المواد المدنية والتجارية؟',
]
for q in tests:
    print('=' * 80)
    print('Q:', q)
    for doc, score in vectordb.similarity_search_with_score(q, k=4):
        m = doc.metadata
        print(f"   {score:.3f}  {m['source']:5} {str(m.get('doc_id'))[:14]:15} "
              f"art={str(m.get('article_no'))[:5]:5} {doc.page_content[:65].strip()}")

## 8. Zip it for download

The store is already safe on Drive. This just makes a single file that is easier to
download and copy into `law-chatbot-langchain/data/chroma_v3` on your laptop.

In [ ]:
!zip -r -q /content/chroma_v3.zip /content/drive/MyDrive/chroma_v3
!cp /content/chroma_v3.zip /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive/chroma_v3.zip
print('saved - download chroma_v3.zip from your Drive')